# Case Study 2: Large Library Screening
Run batched inference over a large library using partitioned processing.
Upload -> Partition -> Featurize each partition -> Infer each partition

In [ ]:
import uuid
import time
from pathlib import Path

from pyds import BaseClient, Data, Featurize, Infer, Partition, Settings

In [ ]:
BASE_URL = "http://deepchem-server"
PROFILE = "test_profile"
PROJECT = "test_project"

repo_root = Path(".").resolve().parent
DATASET_PATH = repo_root / "deepchem_server" / "core" / "tests" / "assets" / "zinc250k.csv"

SCREENING_MODEL = "deepchem://test_profile/test_project/rf_logp_5a4c5f9f"
N_PARTITIONS = 25

run_id = uuid.uuid4().hex[:8]
print(f"Run ID: {run_id} — {time.strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
settings = Settings(profile=PROFILE, project=PROJECT, base_url=BASE_URL)

base_client      = BaseClient(settings=settings)
data_client      = Data(settings=settings)
featurize_client = Featurize(settings=settings)
partition_client = Partition(settings=settings)
infer_client     = Infer(settings=settings)

print("healthcheck:", base_client.healthcheck())

## 1. Upload library
Upload the screening CSV so it can be processed in the datastore.

In [ ]:
upload_result = data_client.upload_data(
    file_path=DATASET_PATH,
    filename=f"library_{run_id}.csv",
    description=f"Screening library upload for run {run_id}",
)
library_address = upload_result["dataset_address"]
print("library_address:", library_address)

## 2. Partition
Split the uploaded library into smaller chunks for scalable execution.

In [ ]:
partition_result = partition_client.run(
    dataset_address=library_address,
    n_partition=N_PARTITIONS,
    shuffle=False,
)
partitions = partition_result["partitioned_dataset_addresses"]
print(f"Partitioned into {len(partitions)} parts")
for i, addr in enumerate(partitions):
    print(f"  [{i}] {addr}")

## 3. Featurize + Infer (per partition)
Apply featurization and inference sequentially for each partition.

In [ ]:
prediction_files = []

for i, part_addr in enumerate(partitions):
    featurize_result = featurize_client.run(
        dataset_address=part_addr,
        featurizer="ecfp",
        output=f"library_ecfp_{run_id}_part{i}",
        dataset_column="smiles",
        label_column="logp",
        feat_kwargs={"radius": 2, "size": 1024},
    )
    print(f"[{i}] featurized: {featurize_result['featurized_file_address']}")

    infer_result = infer_client.run(
        model_address=SCREENING_MODEL,
        data_address=part_addr,
        dataset_column="smiles",
        output=f"screening_{run_id}_part{i}",
    )
    pred_address = infer_result["inference_results_address"]
    prediction_files.append(pred_address)
    print(f"[{i}] predictions: {pred_address}")

print(f"\nScreening complete. {len(prediction_files)} prediction files produced.")

In [ ]:
if prediction_files:
    first_pred = prediction_files[0]
    pred_results = data_client.get(first_pred)
    print(f"First prediction file: {first_pred}")
    print("Prediction preview:")
    if hasattr(pred_results, "head"):
        display(pred_results.head())
    else:
        print(pred_results)
else:
    print("No prediction files generated.")